# 03 - Results Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-pipeline/blob/main/notebooks/03_results_analysis_colab.ipynb)

Loads `screening_results.csv` / `summary.json` produced by
`crypticip screen`, ranks candidates, and plots score distributions,
tier counts, and a feature heatmap for the top 25.

If you don't have a completed screen yet, the bootstrap section
includes an option to run a tiny screen on-the-fly so the rest of the
notebook has something to plot.


## Run this first — fresh-Colab setup

Configure the variables below, then run every cell in this section in
order. Re-running is safe: the clone is updated in place and pip
installs are idempotent.

- `REPO_URL` / `BRANCH` — where to fetch the code from.
- `PROJECT_DIR` — where to clone into (`/content/...` lives on the
  Colab VM and is wiped between sessions).
- `MOUNT_DRIVE` — set `True` to mount Google Drive so outputs persist.
- `DRIVE_RESULTS` — if mounting, where on Drive to put `results/`.


In [ ]:
REPO_URL    = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
BRANCH      = 'main'
PROJECT_DIR = '/content/cryptic-ip-pipeline'
MOUNT_DRIVE = False                  # True to mount Google Drive
DRIVE_ROOT  = '/content/drive'
DRIVE_RESULTS = '/content/drive/MyDrive/crypticip/results'  # used only if MOUNT_DRIVE


In [ ]:
import os, subprocess, sys, pathlib

project = pathlib.Path(PROJECT_DIR)
project.parent.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

if (project / '.git').exists():
    sh(['git', '-C', str(project), 'fetch', '--quiet', 'origin', BRANCH])
    sh(['git', '-C', str(project), 'checkout', BRANCH])
    sh(['git', '-C', str(project), 'reset', '--hard', f'origin/{BRANCH}'])
else:
    sh(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, str(project)])

os.chdir(project)
print('cwd:', os.getcwd())
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat'])


In [ ]:
# Optional: mount Drive and redirect results/ onto it so outputs survive
# a runtime swap. Safe to re-run; pre-existing local results/ is backed
# up to a timestamped sibling rather than deleted.
import datetime, pathlib, shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)
    drive_path = pathlib.Path(DRIVE_RESULTS)
    drive_path.mkdir(parents=True, exist_ok=True)
    target = pathlib.Path(PROJECT_DIR) / 'results'
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        backup = target.with_name(f'results.local_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}')
        shutil.move(str(target), str(backup))
        print('existing results/ backed up to:', backup)
    target.symlink_to(drive_path, target_is_directory=True)
    print('results/ ->', drive_path)
else:
    print('MOUNT_DRIVE=False; outputs will live on the Colab VM only.')


In [ ]:
# Verify the CLI is wired up correctly.
!crypticip --version
!crypticip check-env
!crypticip list-configs


## Locate (or generate) screening output

This notebook looks for `results/screening/{ORGANISM}/screening_results.csv`.
If you mounted Drive in the bootstrap and pointed `DRIVE_RESULTS` at a
directory that already has a screen, it will be picked up automatically.

If no screening output exists, set `RUN_SMOKE_SCREEN = True` below to
run a small synthetic screen so the rest of the notebook is functional.


In [ ]:
import pathlib, pandas as pd, matplotlib.pyplot as plt

ORGANISM = 'yeast'           # 'yeast' / 'human' / 'dictyostelium'
RUN_SMOKE_SCREEN = False     # set True to materialise a tiny synthetic screen
screen_dir = pathlib.Path(f'results/screening/{ORGANISM}')
csv_path   = screen_dir / 'screening_results.csv'
print('looking for:', csv_path, '(exists:', csv_path.exists(), ')')


In [ ]:
import shutil, yaml, subprocess

if not csv_path.exists() and RUN_SMOKE_SCREEN:
    SMOKE = pathlib.Path('/content/crypticip_smoke')
    PROT  = SMOKE / 'data/proteomes/yeast'
    PROT.mkdir(parents=True, exist_ok=True)
    src = pathlib.Path('tests/fixtures/tiny.pdb')
    for u in ['P00001','P00002','P00003']:
        shutil.copyfile(src, PROT / f'AF-{u}-F1-model_v4.pdb')
    cfg = SMOKE / 'smoke_config.yaml'
    cfg.write_text(yaml.safe_dump({'paths': {
        'data_dir':      str(SMOKE / 'data'),
        'proteomes_dir': str(SMOKE / 'data' / 'proteomes'),
        'cache_dir':     str(SMOKE / 'cache'),
        'results_dir':   'results',
        'reports_dir':   'results/reports',
        'screening_dir': 'results/screening',
        'experimental_dir': 'results/experimental',
    }}))
    subprocess.run(['crypticip', 'index-proteome', '--organism', 'yeast',
                    '--config', str(cfg)], check=False)
    subprocess.run(['crypticip', 'screen', '--organism', 'yeast', '--mode', 'fast',
                    '--workers', '1', '--limit', '3', '--config', str(cfg)],
                   check=False)
elif not csv_path.exists():
    print('No screening_results.csv. Either run `crypticip screen --organism', ORGANISM,
          '` first (see 02_yeast_screening_colab.ipynb) or set RUN_SMOKE_SCREEN=True above.')


## Load + preview


In [ ]:
if not csv_path.exists():
    print('still no screening_results.csv — skipping the rest of the notebook')
    df = pd.DataFrame()
else:
    df = pd.read_csv(csv_path)
    print('rows:', len(df), 'cols:', list(df.columns))
    df.head()


## Rank candidates by composite score


In [ ]:
if len(df) and 'score' in df.columns:
    ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
    cols = [c for c in ['uniprot_id','name','score','tier','volume_A3','depth_A',
                        'potential_kT_e'] if c in ranked.columns]
    ranked[cols].head(25)


## Score distribution


In [ ]:
if len(df) and 'score' in df.columns:
    fig, ax = plt.subplots(figsize=(6,4))
    df['score'].dropna().plot.hist(bins=40, ax=ax)
    ax.set_xlabel('score'); ax.set_ylabel('count')
    ax.set_title(f'{ORGANISM} score distribution (n={len(df)})')
    plt.tight_layout(); plt.show()


## Tier counts


In [ ]:
if len(df) and 'tier' in df.columns:
    counts = df['tier'].value_counts().reindex(
        ['tier1','tier2','tier3','tier4']).fillna(0).astype(int)
    print(counts.to_string())
    counts.plot.bar(figsize=(5,3), rot=0)
    plt.tight_layout(); plt.show()


## Feature heatmap for top 25 candidates


In [ ]:
if len(df) >= 5 and 'score' in df.columns:
    feats = [c for c in ['volume_A3','depth_A','potential_kT_e',
                         'basic_residue_count','conservation_score']
             if c in df.columns]
    if feats:
        top = df.sort_values('score', ascending=False).head(25)
        import numpy as np
        z = (top[feats] - top[feats].mean()) / top[feats].std(ddof=0).replace(0, 1)
        fig, ax = plt.subplots(figsize=(7, max(3, 0.3*len(top))))
        im = ax.imshow(z.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
        ax.set_xticks(range(len(feats))); ax.set_xticklabels(feats, rotation=30, ha='right')
        ax.set_yticks(range(len(top))); ax.set_yticklabels(top.get('uniprot_id', top.index))
        plt.colorbar(im); plt.tight_layout(); plt.show()


## Export ranked CSV + download


In [ ]:
OUT_DIR = pathlib.Path('results/reports/analysis'); OUT_DIR.mkdir(parents=True, exist_ok=True)
if len(df) and 'score' in df.columns:
    out_csv = OUT_DIR / f'{ORGANISM}_ranked.csv'
    df.sort_values('score', ascending=False).to_csv(out_csv, index=False)
    print('wrote:', out_csv)
    try:
        from google.colab import files
        files.download(str(out_csv))
    except Exception as e:
        print('skipping files.download (not on Colab):', e)


## Summary

- This notebook is purely analytic — it does **not** re-run any expensive
  step (unless you set `RUN_SMOKE_SCREEN=True`).
- You can iterate on plots without touching the screening data.
- Outputs land in `results/reports/analysis/`. If you mounted Drive,
  they're already persisted.
